# Stage 1 — Soft-normalize annotator corrections

Map each completed top-1 correction onto the 16-class MIDAS `y16` label.

1. Rule-based normalization (abbreviations, synonyms → canonical y16)
2. Join MIDAS pathology ground truth by `case_id`
3. Where rule ≠ pathology, optionally ask GPT-4o if the free text is a reasonable match
4. Write `data/normed/edits_soft_normalized.csv`

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "editing" / "shared"))
sys.path.insert(0, str(Path(".").resolve()))

import config as C
from data_engineering.normalize_dx import normalize
from data_engineering.parse_annotations import filter_records, load_records
from gt import load as load_gt

In [2]:
OUT_DIR = REPO_ROOT / "data" / "normed"
OUT_CSV = OUT_DIR / "edits_soft_normalized.csv"
LLM_CACHE = OUT_DIR / "llm_judgments.json"

USE_LLM = True   # False = rule-based labels only, no API calls
# USE_LLM=True always calls GPT-4o on mismatches. A previous llm_judgments.json is not reused.

## 1. Build rows — rule-based normalization + ground truth

In [3]:
import pandas as pd

gt = load_gt()
records = filter_records(
    load_records(C.ANNOTATIONS_CSV),
    require_complete=True,
    top1_errors_only=True,
    require_correction=True,
)

rows = []
for rec in records:
    nd = normalize(rec.human_top1_raw)
    rows.append({
        "login_id": rec.login_id,
        "case_id": rec.case_id,
        "model": rec.model,
        "model_top1": rec.model_top1,
        "human_top1_raw": rec.human_top1_raw,
        "human_top1_is_multi": nd.is_multi,
        "human_top1_norm": nd.norm,
        "human_top1_y16_rule": nd.y16,
        "gt_y16": gt.get(rec.case_id),
        "llm_match": None,
        "llm_reason": None,
        "human_top1_y16_soft": None,
    })

df = pd.DataFrame(rows)
print(f"{len(df)} corrections loaded")
df.head()

156 corrections loaded


,login_id,case_id,model,model_top1,human_top1_raw,human_top1_is_multi,human_top1_norm,human_top1_y16_rule,gt_y16,llm_match,llm_reason,human_top1_y16_soft
0,achiou@stanford.edu,1006_combined,GPT-5.3,Inflamed folliculitis,BCC,False,basal cell carcinoma,Basal Cell Carcinoma,Basal Cell Carcinoma,None,None,None
1,achiou@stanford.edu,1016_combined,GPT-5.3,Bacterial folliculitis,Squamous cell carcinoma,False,squamous cell carcinoma,Squamous Cell Carcinoma,Other Benign,None,None,None
2,achiou@stanford.edu,101_combined,DermatoLlama,Nevus,dysplastic nevus,False,dysplastic nevus,Melanocytic Nevus,Melanocytic Nevus,None,None,None
3,achiou@stanford.edu,1038_combined,DermatoLlama,Seborrheic Keratosis,dermatofibroma,False,dermatofibroma,Dermatofibroma,Dermatofibroma,None,None,None
4,achiou@stanford.edu,1038_combined,GPT-5.3,Lichen planus–like keratosis,Dermatofibroma,False,dermatofibroma,Dermatofibroma,Dermatofibroma,None,None,None


## 2. Summary — rule match vs mismatch

In [4]:
n_rule_match = (df["human_top1_y16_rule"] == df["gt_y16"]).sum()
n_no_gt = df["gt_y16"].isna().sum()
n_mismatch = len(df) - n_rule_match - n_no_gt

print(f"Rule matches GT:  {n_rule_match}")
print(f"Mismatch (→ LLM): {n_mismatch}")
print(f"No GT available:  {n_no_gt}")

Rule matches GT:  34
Mismatch (→ LLM): 106
No GT available:  16


### Mismatch rows (these go to GPT-4o if `USE_LLM=True`)

In [5]:
mismatches = df[
    df["gt_y16"].notna() & (df["human_top1_y16_rule"] != df["gt_y16"])
][["case_id", "human_top1_raw", "human_top1_y16_rule", "gt_y16"]]
print(f"{len(mismatches)} mismatches")
mismatches

106 mismatches


,case_id,human_top1_raw,human_top1_y16_rule,gt_y16
1,1016_combined,Squamous cell carcinoma,Squamous Cell Carcinoma,Other Benign
5,1038_combined,pigmented SCC,Squamous Cell Carcinoma,Dermatofibroma
8,1065_combined,squamous cell carcinoma,Squamous Cell Carcinoma,Actinic Keratosis
9,1111_combined,basal cell carcinoma,Basal Cell Carcinoma,Other Benign
14,145_combined,Fibrous papule,Fibrous Papule,Melanocytic Nevus
...,...,...,...,...
148,845_combined,actinic keratosis vs squamous cell carcinoma v...,Actinic Keratosis,Squamous Cell Carcinoma In Situ
149,902_combined,actinic keratosis vs inflamed seborrheic kerat...,Actinic Keratosis,Seborrheic Keratosis
152,970_combined,dysplastic nevus,Melanocytic Nevus,Actinic Keratosis
153,970_combined,melanoma vs dysplastic nevus,Melanoma,Actinic Keratosis


## 3. Apply soft normalization

In [6]:
import json

from llm_judge import judge

# Recompute every run. `seen` only skips a second call for the same
# free text and pathology label inside this run.
seen = {}
n_api = 0

for row in rows:
    rule = row["human_top1_y16_rule"]
    gt_val = row["gt_y16"]

    if gt_val is None:
        row["human_top1_y16_soft"] = rule
        continue

    if rule == gt_val:
        row["llm_match"] = True
        row["human_top1_y16_soft"] = gt_val
        continue

    if not USE_LLM:
        row["human_top1_y16_soft"] = rule
        continue

    key = f"{row['human_top1_raw']}||{gt_val}"
    if key not in seen:
        seen[key] = judge(row["human_top1_raw"], gt_val)
        n_api += 1
    verdict = seen[key]

    row["llm_match"] = verdict.get("match", False)
    row["llm_reason"] = verdict.get("reason", "")
    row["human_top1_y16_soft"] = gt_val if verdict.get("match") else rule

OUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_CACHE.write_text(json.dumps(seen, indent=2, ensure_ascii=False))
print(f"{n_api} LLM API calls")

df = pd.DataFrame(rows)

n_rule_match = (df["human_top1_y16_rule"] == df["gt_y16"]).sum()
n_llm_match = ((df["llm_match"] == True) & (df["human_top1_y16_rule"] != df["gt_y16"])).sum()
n_no_gt = df["gt_y16"].isna().sum()
n_wrong = len(df) - n_rule_match - n_llm_match - n_no_gt

print(f"Rule match:    {n_rule_match}")
print(f"LLM promoted:  {n_llm_match}")
print(f"Genuine error: {n_wrong}")
print(f"No GT:         {n_no_gt}")

80 LLM API calls
Rule match:    34
LLM promoted:  23
Genuine error: 83
No GT:         16


### Inspect LLM-promoted rows

In [7]:
promoted = df[
    (df["llm_match"] == True) & (df["human_top1_y16_rule"] != df["gt_y16"])
][["case_id", "human_top1_raw", "human_top1_y16_rule", "gt_y16", "llm_reason"]]
print(f"{len(promoted)} rows promoted by LLM")
promoted

23 rows promoted by LLM


,case_id,human_top1_raw,human_top1_y16_rule,gt_y16,llm_reason
38,397_combined,SK,Seborrheic Keratosis,Other Benign,SK (seborrheic keratosis) is a benign lesion a...
39,420_combined,dermatofibroma,Dermatofibroma,Other Benign,Dermatofibroma is a benign skin lesion and fal...
40,420_combined,scar,Non-neoplastic,Other Benign,A scar is a benign condition and falls under t...
41,420_combined,scar,Non-neoplastic,Other Benign,A scar is a benign condition and falls under t...
47,573_combined,surface of vascular growth like a hemangioma,Hemangioma,Other Benign,"A hemangioma is a benign vascular growth, whic..."
76,1168_combined,Melanoma in situ,Melanoma,Melanocytic Lesion,Melanoma in situ is a subtype of melanocytic l...
81,143_combined,Lichen simplex chronicus,Non-neoplastic,Other Benign,Lichen simplex chronicus is a benign skin cond...
82,144_combined,Lentigo,Melanocytic Lesion,Other Benign,"Lentigo is a benign pigmented lesion, which fa..."
83,144_combined,Lentigo,Melanocytic Lesion,Other Benign,"Lentigo is a benign pigmented lesion, which fa..."
85,151_combined,Squamous cell carcinoma in situ,Squamous Cell Carcinoma In Situ,Squamous Cell Carcinoma,Squamous cell carcinoma in situ is a recognize...


## 4. Write CSV

In [8]:
import csv

COLUMNS = [
    "login_id", "case_id", "model",
    "model_top1", "human_top1_raw", "human_top1_is_multi",
    "human_top1_norm", "human_top1_y16_rule", "gt_y16",
    "llm_match", "llm_reason",
    "human_top1_y16_soft",
]

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

print(f"Wrote {len(rows)} rows → {OUT_CSV}")

Wrote 156 rows → /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/data/normed/edits_soft_normalized.csv
